# Filter Job Descriptions Mentioning AI-Related Keywords

Scans all parquet files (20M+ records) and extracts job descriptions
that mention AI-related terms using **exact word-boundary matching** (case-insensitive).

**Strategy:**
- DuckDB `regexp_matches` directly on parquet files (no pandas load)
- Single-pass filter with `COPY TO` for output
- Summary statistics by company and keyword

In [ ]:
# =============================================================================
# CONFIGURATION & IMPORTS
# =============================================================================

import duckdb
import pandas as pd
import os
import re
import json
from glob import glob
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

# =====================================================================
# PATHS
# =====================================================================
INPUT_FOLDER = '../JD'
OUTPUT_DIR = '../data'
OUTPUT_PARQUET = f'{OUTPUT_DIR}/ai_keyword_filtered_jds.parquet'
OUTPUT_CSV = f'{OUTPUT_DIR}/ai_keyword_filtered_jds.csv'
STATS_PATH = f'{OUTPUT_DIR}/ai_keyword_filter_stats.json'

# Column names
JD_TEXT_COLUMN = 'job_description'
COMPANY_COLUMN = 'company'
COLUMNS_TO_KEEP = ['company', 'job_description', 'rcid', 'onet_code', 'post_date']

# =====================================================================
# AI KEYWORDS LIST
# =====================================================================
AI_KEYWORDS = [
    # Core AI terms
    'AI', 'artificial intelligence',
    # Machine learning
    'machine learning', 'ML', 'deep learning', 'DL',
    'neural network', 'neural networks',
    # NLP / LLM
    'NLP', 'natural language processing',
    'LLM', 'large language model', 'large language models',
    # Generative AI & specific models
    'generative AI', 'GenAI', 'gen AI',
    'GPT', 'ChatGPT', 'BERT',
    'transformer', 'transformers',
    'diffusion model', 'diffusion models',
    # Computer vision
    'computer vision', 'image recognition',
    # Other AI subfields
    'reinforcement learning',
    'speech recognition',
    'autonomous systems',
    # Applied AI
    'predictive modeling', 'predictive analytics',
    'recommendation system', 'recommendation systems',
    'robotic process automation', 'RPA',
    'intelligent automation',
    'cognitive computing',
    'knowledge graph', 'knowledge graphs',
    # LLM-era terms
    'prompt engineering',
    'retrieval augmented generation', 'RAG',
    'fine-tuning', 'fine tuning',
    'embeddings',
    'vector database', 'vector databases',
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Configuration ready. {len(AI_KEYWORDS)} AI keywords defined.')

In [ ]:
# =============================================================================
# BUILD REGEX PATTERN
# =============================================================================

def build_regex_pattern(keywords):
    """Build a POSIX-compatible regex with word boundaries, case-insensitive."""
    escaped = [re.escape(kw) for kw in keywords]
    # Sort by length descending so longer phrases match before shorter ones
    escaped.sort(key=len, reverse=True)
    alternation = '|'.join(escaped)
    pattern = f'(?i)\\b({alternation})\\b'
    return pattern

REGEX_PATTERN = build_regex_pattern(AI_KEYWORDS)

print(f'Regex pattern length: {len(REGEX_PATTERN)} chars')
print(f'Pattern preview: {REGEX_PATTERN[:200]}...')

# Quick sanity check
import re as _re
p = _re.compile(REGEX_PATTERN)
assert p.search('We use AI for automation'), 'Should match AI'
assert p.search('machine learning engineer'), 'Should match machine learning'
assert not p.search('email sending service'), 'Should NOT match email'
assert not p.search('retail operations'), 'Should NOT match retail'
print('Sanity checks passed.')

In [ ]:
# =============================================================================
# DISCOVER INPUT FILES
# =============================================================================

input_files = sorted(glob(f'{INPUT_FOLDER}/*.parquet'))
print(f'Found {len(input_files)} parquet files in {INPUT_FOLDER}')

if len(input_files) == 0:
    raise FileNotFoundError(f'No parquet files found in {INPUT_FOLDER}')

# Show first few files and total size
for f in input_files[:5]:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f'  {os.path.basename(f):40s} ({size_mb:.1f} MB)')
if len(input_files) > 5:
    print(f'  ... and {len(input_files) - 5} more files')

total_size_gb = sum(os.path.getsize(f) for f in input_files) / (1024**3)
print(f'\nTotal size: {total_size_gb:.2f} GB')

# Verify required columns exist
conn = duckdb.connect()
schema = conn.execute(f"DESCRIBE SELECT * FROM read_parquet('{input_files[0]}')").fetchdf()
columns = set(schema['column_name'].tolist())
for col in [JD_TEXT_COLUMN, COMPANY_COLUMN]:
    status = 'OK' if col in columns else 'NOT FOUND'
    print(f'  Column {col}: {status}')
conn.close()

print(f'\nAll columns: {sorted(columns)}')

In [ ]:
# =============================================================================
# FILTER JOB DESCRIPTIONS WITH DUCKDB
# =============================================================================

print('='*70)
print('Filtering job descriptions for AI-related keywords')
print('='*70)
print('This may take several minutes for large datasets...\n')

conn = duckdb.connect()
conn.execute('SET memory_limit = "4GB"')
conn.execute('SET threads TO 4')

files_pattern = f'{INPUT_FOLDER}/*.parquet'
columns_sql = ', '.join(COLUMNS_TO_KEEP)

# Escape single quotes in regex pattern for SQL
sql_pattern = REGEX_PATTERN.replace("'", "''")

filter_query = f"""
SELECT {columns_sql}
FROM read_parquet('{files_pattern}', union_by_name=true)
WHERE {JD_TEXT_COLUMN} IS NOT NULL
  AND regexp_matches({JD_TEXT_COLUMN}, '{sql_pattern}')
"""

# Write directly to parquet (no Python memory needed)
copy_query = f"""
COPY (
    {filter_query}
) TO '{OUTPUT_PARQUET}' (FORMAT PARQUET, COMPRESSION ZSTD)
"""

conn.execute(copy_query)

# Count results
total_matched = conn.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PARQUET}')"
).fetchone()[0]

output_size_mb = os.path.getsize(OUTPUT_PARQUET) / (1024 * 1024)
print(f'Matched {total_matched:,} job descriptions mentioning AI keywords')
print(f'Output parquet: {OUTPUT_PARQUET} ({output_size_mb:.1f} MB)')

conn.close()

In [ ]:
# =============================================================================
# EXPORT AS CSV
# =============================================================================

conn = duckdb.connect()
conn.execute(f"""
COPY (
    SELECT * FROM read_parquet('{OUTPUT_PARQUET}')
) TO '{OUTPUT_CSV}' (FORMAT CSV, HEADER TRUE)
""")
conn.close()

csv_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
print(f'Exported CSV: {OUTPUT_CSV} ({csv_size_mb:.1f} MB)')

In [ ]:
# =============================================================================
# SUMMARY STATISTICS
# =============================================================================

conn = duckdb.connect()

# Total counts
unique_companies = conn.execute(
    f"SELECT COUNT(DISTINCT {COMPANY_COLUMN}) FROM read_parquet('{OUTPUT_PARQUET}')"
).fetchone()[0]

print('='*70)
print('SUMMARY STATISTICS')
print('='*70)
print(f'Total matched JDs:    {total_matched:,}')
print(f'Unique companies:     {unique_companies:,}')

# Top 30 companies by AI JD count
print(f'\nTop 30 companies by AI-mentioning job descriptions:')
print('-'*60)
company_stats = conn.execute(f"""
    SELECT {COMPANY_COLUMN}, COUNT(*) as ai_jd_count
    FROM read_parquet('{OUTPUT_PARQUET}')
    WHERE {COMPANY_COLUMN} IS NOT NULL
    GROUP BY {COMPANY_COLUMN}
    ORDER BY ai_jd_count DESC
    LIMIT 30
""").fetchdf()

for i, row in company_stats.iterrows():
    print(f"  {i+1:2d}. {row[COMPANY_COLUMN]:45s} {row['ai_jd_count']:>8,}")

# Per-keyword frequency (queries on already-filtered output, so fast)
print(f'\nKeyword frequency (across {total_matched:,} matched JDs):')
print('-'*60)
keyword_counts = []
for kw in AI_KEYWORDS:
    escaped = re.escape(kw).replace("'", "''")
    kw_pattern = f"(?i)\\b{escaped}\\b"
    count = conn.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{OUTPUT_PARQUET}')
        WHERE regexp_matches({JD_TEXT_COLUMN}, '{kw_pattern}')
    """).fetchone()[0]
    keyword_counts.append((kw, count))

keyword_df = pd.DataFrame(keyword_counts, columns=['keyword', 'count'])
keyword_df = keyword_df.sort_values('count', ascending=False).reset_index(drop=True)

for i, row in keyword_df.iterrows():
    pct = row['count'] / total_matched * 100 if total_matched > 0 else 0
    print(f"  {row['keyword']:40s} {row['count']:>8,}  ({pct:5.1f}%)")

# Distribution by quarter
print(f'\nDistribution by quarter:')
print('-'*60)
temporal_stats = conn.execute(f"""
    SELECT
        CAST(YEAR(CAST(post_date AS DATE)) AS VARCHAR) || 'Q' ||
        CAST(QUARTER(CAST(post_date AS DATE)) AS VARCHAR) AS quarter,
        COUNT(*) as count
    FROM read_parquet('{OUTPUT_PARQUET}')
    WHERE post_date IS NOT NULL
    GROUP BY quarter
    ORDER BY quarter
""").fetchdf()

for _, row in temporal_stats.iterrows():
    print(f"  {row['quarter']:10s} {row['count']:>8,}")

# Distribution by ONET code (top 20)
print(f'\nTop 20 ONET codes:')
print('-'*60)
onet_stats = conn.execute(f"""
    SELECT onet_code, COUNT(*) as count
    FROM read_parquet('{OUTPUT_PARQUET}')
    WHERE onet_code IS NOT NULL
    GROUP BY onet_code
    ORDER BY count DESC
    LIMIT 20
""").fetchdf()

for _, row in onet_stats.iterrows():
    print(f"  {row['onet_code']:15s} {row['count']:>8,}")

conn.close()

In [ ]:
# =============================================================================
# VISUALIZATIONS
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# 1. Top 20 keywords
ax = axes[0]
top_kw = keyword_df.head(20)
ax.barh(top_kw['keyword'][::-1], top_kw['count'][::-1], edgecolor='black', alpha=0.7)
ax.set_xlabel('Number of JDs')
ax.set_title('Top 20 AI Keywords by Frequency')
ax.grid(axis='x', alpha=0.3)

# 2. Top 20 companies
ax = axes[1]
top_co = company_stats.head(20)
ax.barh(top_co[COMPANY_COLUMN][::-1], top_co['ai_jd_count'][::-1],
        edgecolor='black', alpha=0.7, color='#e67e22')
ax.set_xlabel('Number of AI-mentioning JDs')
ax.set_title('Top 20 Companies by AI JD Count')
ax.grid(axis='x', alpha=0.3)

# 3. AI JD count by quarter
ax = axes[2]
x_labels = temporal_stats['quarter'].tolist()
tick_step = max(1, len(x_labels) // 12)
ax.bar(range(len(x_labels)), temporal_stats['count'], edgecolor='black', alpha=0.7, color='#27ae60')
ax.set_xticks(range(0, len(x_labels), tick_step))
ax.set_xticklabels([x_labels[i] for i in range(0, len(x_labels), tick_step)],
                   rotation=45, ha='right')
ax.set_xlabel('Quarter')
ax.set_ylabel('Number of JDs')
ax.set_title('AI-Mentioning JDs Over Time')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# FULL COMPANY LIST THAT MENTION ANY AI KEYWORD
# =============================================================================

conn = duckdb.connect()

all_companies = conn.execute(f"""
    SELECT {COMPANY_COLUMN}, COUNT(*) as ai_jd_count
    FROM read_parquet('{OUTPUT_PARQUET}')
    WHERE {COMPANY_COLUMN} IS NOT NULL
    GROUP BY {COMPANY_COLUMN}
    ORDER BY ai_jd_count DESC
""").fetchdf()

conn.close()

print('='*70)
print(f'ALL COMPANIES MENTIONING AI KEYWORDS ({len(all_companies):,} companies)')
print('='*70)
print(f'{"#":>5s}  {"Company":<50s} {"AI JDs":>8s}')
print('-'*70)
for i, row in all_companies.iterrows():
    print(f'{i+1:5d}  {row[COMPANY_COLUMN]:<50s} {row["ai_jd_count"]:>8,}')

print(f'\nTotal: {len(all_companies):,} unique companies')

In [ ]:
# =============================================================================
# FIRST AI-MENTIONING POST PER COMPANY (date, JD text, matched keywords)
# =============================================================================

conn = duckdb.connect()

# Get the earliest post per company
first_posts = conn.execute(f"""
    WITH ranked AS (
        SELECT
            {COMPANY_COLUMN},
            post_date,
            {JD_TEXT_COLUMN},
            ROW_NUMBER() OVER (
                PARTITION BY {COMPANY_COLUMN}
                ORDER BY CAST(post_date AS DATE) ASC
            ) AS rn
        FROM read_parquet('{OUTPUT_PARQUET}')
        WHERE {COMPANY_COLUMN} IS NOT NULL
          AND post_date IS NOT NULL
    )
    SELECT {COMPANY_COLUMN}, post_date, {JD_TEXT_COLUMN}
    FROM ranked
    WHERE rn = 1
    ORDER BY CAST(post_date AS DATE) ASC
""").fetchdf()

conn.close()

# Find matched keywords for each JD using Python regex
pattern = re.compile(REGEX_PATTERN)

def find_matched_keywords(text):
    """Return sorted unique list of AI keywords found in text."""
    if not isinstance(text, str):
        return []
    matches = pattern.findall(text)
    # findall returns the captured group; normalize to lowercase for dedup
    unique = sorted(set(m for m in matches), key=str.lower)
    return unique

first_posts['matched_keywords'] = first_posts[JD_TEXT_COLUMN].apply(find_matched_keywords)

# Display results
JD_PREVIEW_LEN = 200

print('=' * 100)
print(f'FIRST AI-MENTIONING POST PER COMPANY ({len(first_posts):,} companies)')
print('=' * 100)

for i, row in first_posts.iterrows():
    company = row[COMPANY_COLUMN]
    date = str(row['post_date'])[:10]
    jd_preview = str(row[JD_TEXT_COLUMN])[:JD_PREVIEW_LEN].replace('\n', ' ')
    keywords = ', '.join(row['matched_keywords'])

    print(f'\n{"─"*100}')
    print(f'  #{i+1}  Company:  {company}')
    print(f'       Date:     {date}')
    print(f'       Keywords: {keywords}')
    print(f'       JD:       {jd_preview}...')

print(f'\n{"="*100}')
print(f'Total: {len(first_posts):,} companies with their first AI-mentioning post')

In [ ]:
# =============================================================================
# SAVE STATISTICS AS JSON
# =============================================================================

stats_summary = {
    'total_matched_jds': int(total_matched),
    'unique_companies': int(unique_companies),
    'keywords_used': AI_KEYWORDS,
    'keyword_frequencies': keyword_df.to_dict('records'),
    'top_companies': company_stats.to_dict('records'),
    'output_files': {
        'parquet': OUTPUT_PARQUET,
        'csv': OUTPUT_CSV,
    }
}

with open(STATS_PATH, 'w') as f:
    json.dump(stats_summary, f, indent=2)

print(f'Statistics saved to {STATS_PATH}')
print(f'\n{"="*70}')
print('DONE')
print(f'{"="*70}')
print(f'Output parquet: {OUTPUT_PARQUET}')
print(f'Output CSV:     {OUTPUT_CSV}')
print(f'Statistics:     {STATS_PATH}')
print(f'\nTotal matched:  {total_matched:,} JDs from {unique_companies:,} companies')